In [1]:
%load_ext autoreload
%autoreload 2

In [7]:
from hydra import initialize, initialize_config_module, initialize_config_dir, compose
import hydra
from omegaconf import OmegaConf
from models.components.codestral import Codestral
import os

os.environ["HYDRA_FULL_ERROR"] = "1"


In [18]:
from pathlib import Path

dir = Path("../configs").absolute()
with initialize(version_base=None, config_path="../configs"):
    cfg = compose("model/components/codestral.yaml")#, overrides=["+model_path=/share/homes/gerel/gramscore/weights/Codestral-22B-v0.1"])
    print(OmegaConf.to_yaml(cfg))

model:
  components:
    quantizer_config:
      _target_: transformers.BitsAndBytesConfig
      load_in_4bit: true
      bnb_4bit_quant_type: nf4
      bnb_4bit_use_double_quant: true
      bnb_4bit_compute_dtype: bfloat16
    _target_: src.models.components.codestral.Codestral
    model_path: mistralai/Codestral-22B-v0.1



/home/gerel/miniconda3/envs/gramscore/lib/python3.12/site-packages/hydra/_internal/defaults_list.py:251: UserWarning: In 'model/components/codestral.yaml': Defaults list is missing `_self_`. See https://hydra.cc/docs/1.2/upgrades/1.0_to_1.1/default_composition_order for more information
  warnings.warn(msg, UserWarning)


In [19]:
import torch
import gc

torch.cuda.empty_cache()
gc.collect()

2999

In [20]:
cfg.model.components

{'quantizer_config': {'_target_': 'transformers.BitsAndBytesConfig', 'load_in_4bit': True, 'bnb_4bit_quant_type': 'nf4', 'bnb_4bit_use_double_quant': True, 'bnb_4bit_compute_dtype': 'bfloat16'}, '_target_': 'src.models.components.codestral.Codestral', 'model_path': 'mistralai/Codestral-22B-v0.1'}

In [21]:
model: Codestral = hydra.utils.instantiate(cfg.model.components)
model

Loading checkpoint shards:   0%|          | 0/9 [00:04<?, ?it/s]


KeyboardInterrupt: 

In [15]:
from transformers import TextStreamer

In [17]:
text = "Hello my name is"
# text = "Generate a Lilypond file of 'Jingle Bells'"
print(model.tokenizer)
inputs = model.tokenizer(text, return_tensors="pt")

streamer = TextStreamer(model.tokenizer)
outputs = model.model.generate(**inputs, max_new_tokens=5, temperature=0.3, do_sample=True, streamer=streamer)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


LlamaTokenizerFast(name_or_path='mistralai/Codestral-22B-v0.1', vocab_size=32768, model_max_length=1000000000000000019884624838656, is_fast=True, padding_side='left', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>'}, clean_up_tokenization_spaces=False),  added_tokens_decoder={
	0: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("[INST]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=False),
	4: AddedToken("[/INST]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=False),
	5: AddedToken("[TOOL_CALLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=False),
	6: AddedToken("[AVAILABLE_TOOLS]


```
\version "2.22.1"

\header {
  title = "Jingle Bells"
  composer = ""
}

\score {
  \new Staff {
    \key c \major
    \time 4/4
    \tempo "Moderato"

    \relative c' {
      \repeat volta 2 {
        c8 d e d c d e g g a g f e d c
        c8 d e d c d e g g a g f e d c
        e8 e e d d d e g g g a a g f e d
        c8 d e d c d e g g a g f e d c
      }
    }
  }

  \layout { }
}
```